# P1 complete-cell response-distribution audit (post-primary)

Run on **CPU**. This reads all 24 private P1 units, verifies their hashes and their agreement with the frozen public decision, and writes only source-ID-free descriptive summaries to a separate Drive folder. It never loads a checkpoint, reruns the P1 decision, changes a threshold, or reselects the three representative examples. The results are post-primary, not a new paper gate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, subprocess, sys

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    dirty = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Colab repository has local changes; preserve them before updating.')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements/colab-base.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO)], check=True)
print('Code commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
SOURCE_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p1_shape_v1')
OUTPUT_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p1_response_distribution_v1')
if not (SOURCE_ROOT / 'units').is_dir():
    raise RuntimeError('Private P1 units are missing from Google Drive; do not use smoke or public summary files.')
subprocess.run([sys.executable, '-m', 'covfaith_diagnostics.response_distribution',
                '--repo', str(REPO), '--source', str(SOURCE_ROOT),
                '--output', str(OUTPUT_ROOT)], cwd=REPO, check=True)
report = json.loads((OUTPUT_ROOT / 'response_distribution_report.json').read_text())
for cell, values in report['cells'].items():
    print(json.dumps({
        'cell': cell, 'n': values['n'],
        'rgr_median': values['rgr']['median'],
        'full_l1_ratio_median': values['full_l1_ratio']['median'],
        'onset_median_steps': values['onset_error_steps']['median'],
        'onset_q90_steps': values['onset_error_steps']['q90'],
        'peak_median_steps': values['peak_time_error_steps']['median'],
        'peak_q90_steps': values['peak_time_error_steps']['q90'],
        'onset_within_one_step_count': values['onset_within_one_step_count'],
        'peak_within_one_step_count': values['peak_within_one_step_count'],
    }))
print('Private report:', OUTPUT_ROOT / 'response_distribution_report.json')

Send the eight printed JSON lines and the report status/inventory hash to Codex. Do not upload the private `.npz` files or individual series IDs. The paper will remain unchanged until the complete descriptive output is reviewed.